In [1]:
import gzip
import pandas as pd
import numpy as np

In [2]:
print("Extracting Patient Labels...")
labels = []
with gzip.open("./GSE13159_series_matrix.txt.gz", "rt") as f:
    for line in f:
        # Relaxed search condition
        if line.startswith("!Sample_characteristics_ch1") and ("leukemia" in line.lower() or "healthy" in line.lower()):
            parts = line.strip().split('\t')[1:]
            labels = [p.replace('"', '').lower() for p in parts]
            break

Extracting Patient Labels...


In [3]:
print("Loading Expression Matrix (This takes ~30 seconds)...")
expression_data = pd.read_csv("./GSE13159_series_matrix.txt.gz", sep="\t", comment="!", index_col=0)

df = expression_data.T
df['Target_Label'] = labels

Loading Expression Matrix (This takes ~30 seconds)...


In [4]:
print("Filtering for the 'Big Four' Leukemias + Healthy...")
# We use regex boundaries (\b) so we don't accidentally match parts of other words
conditions = [
    df['Target_Label'].str.contains('healthy', regex=False),
    df['Target_Label'].str.contains(r'\ball\b|-all', regex=True), # Catches t-all, c-all, and standalone 'all'
    df['Target_Label'].str.contains(r'\baml\b', regex=True),
    df['Target_Label'].str.contains(r'\bcll\b', regex=True),
    df['Target_Label'].str.contains(r'\bcml\b', regex=True)
]

Filtering for the 'Big Four' Leukemias + Healthy...


In [5]:
# Map to integers: 0=Healthy, 1=ALL, 2=AML, 3=CLL, 4=CML
choices = [0, 1, 2, 3, 4]
df['Target'] = np.select(conditions, choices, default=-1)

# Drop the rare subtypes we don't want (where Target remains -1)
clean_df = df[df['Target'] != -1].copy()
clean_df = clean_df.drop(columns=['Target_Label'])

In [7]:
print(f"Success! Final Dataset Shape: {clean_df.shape[0]} patients, {clean_df.shape[1]-1} genes.")
print("\nClass breakdown:")
print(clean_df['Target'].value_counts())

clean_df.to_csv("Leukemia_5Class_Dataset.csv", index=False)
print("\nSaved as Leukemia_5Class_Dataset.csv. Ready for ElasticNet.")

Success! Final Dataset Shape: 1890 patients, 54675 genes.

Class breakdown:
Target
1    750
2    542
3    448
4     76
0     74
Name: count, dtype: int64

Saved as Leukemia_5Class_Dataset.csv. Ready for ElasticNet.
